In [1]:
from pathlib import Path
import re, json
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_selection import VarianceThreshold
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


2026-02-10 01:52:34.078506: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-10 01:52:34.234558: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-10 01:52:34.273399: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-10 01:52:34.876819: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; 

In [13]:
import zipfile

archivo_zip = "Proyecto_Grado_DataSets/DDoS-HTTP_Flood-.pcap.zip"
directorio_destino = "/home/sajaimesp/Tesis_Sistemas/Data_Sets/Proyecto_Grado_DataSets"

with zipfile.ZipFile(archivo_zip, 'r') as zip_ref:
    zip_ref.extractall(directorio_destino)

print(f"Archivo {archivo_zip} descomprimido exitosamente.")


Archivo Proyecto_Grado_DataSets/DDoS-HTTP_Flood-.pcap.zip descomprimido exitosamente.


In [14]:
DATA_DIR = Path("/home/sajaimesp/Tesis_Sistemas/Data_Sets/Proyecto_Grado_DataSets")  # <-- AJUSTA

CSV_GLOB = "*.pcap.csv"    # asume que ya convertiste pcap -> csv
files = sorted(DATA_DIR.glob(CSV_GLOB))

print("DATA_DIR:", DATA_DIR.resolve())
print("CSV encontrados:", len(files))
for f in files[:20]:
    print("  -", f.name)


DATA_DIR: /home/sajaimesp/Tesis_Sistemas/Data_Sets/Proyecto_Grado_DataSets
CSV encontrados: 62
  - BenignTraffic.pcap.csv
  - BenignTraffic1.pcap.csv
  - BenignTraffic2.pcap.csv
  - BenignTraffic3.pcap.csv
  - DDoS-ACK_Fragmentation.pcap.csv
  - DDoS-ACK_Fragmentation1.pcap.csv
  - DDoS-ACK_Fragmentation10.pcap.csv
  - DDoS-ACK_Fragmentation11.pcap.csv
  - DDoS-ACK_Fragmentation12.pcap.csv
  - DDoS-ACK_Fragmentation2.pcap.csv
  - DDoS-ACK_Fragmentation3.pcap.csv
  - DDoS-ACK_Fragmentation4.pcap.csv
  - DDoS-ACK_Fragmentation5.pcap.csv
  - DDoS-ACK_Fragmentation6.pcap.csv
  - DDoS-ACK_Fragmentation7.pcap.csv
  - DDoS-ACK_Fragmentation8.pcap.csv
  - DDoS-ACK_Fragmentation9.pcap.csv
  - DDoS-HTTP_Flood-.pcap.csv
  - DDoS-ICMP_Flood.pcap.csv
  - DDoS-ICMP_Flood1.pcap.csv


In [25]:
def infer_label_from_filename(name: str) -> str:
    n = name.lower()

    # quitar extensión .csv si existe y normalizar separadores
    n = n.replace(".csv", "")
    n = n.replace("&", "_and_")
    n = re.sub(r"[\s]+", "_", n)
    n = n.replace("-", "_")
    n = re.sub(r"_+", "_", n)

    # patrones
    if "dns_spoofing" in n:
        return "dns_spoofing"

    if "sqlinjection" in n or "sql_injection" in n:
        return "sql_injection_uploading"

    # DDoS
    if "ddos_ack_fragmentation" in n:
        return "ddos_ack_fragmentation"
    if "ddos_icmp_flood" in n:
        return "ddos_icmp_flood"
    if "ddos_http_flood" in n:
        return "ddos_http_flood"
    if "ddos_tcp_flood" in n:
        return "ddos_tcp_flood"

    # DoS
    if "dos_http_flood" in n:
        return "dos_http_flood"
    if "dos_syn_flood" in n:
        return "dos_syn_flood"
    if "dos_tcp_flood" in n:
        return "dos_tcp_flood"

    # benigno (si existiera)
    if "benign" in n or "normal" in n:
        return "benign"

    return None  # <- importante: None en vez de "unknown"



rows = []
for f in files:
    lab = infer_label_from_filename(f.name)
    rows.append({"file": str(f), "fname": f.name, "label": lab})

df_lab = pd.DataFrame(rows)

print("Distribución labels (incluye None):")
print(df_lab["label"].value_counts(dropna=False))

print("\nEjemplos con label None (si hay):")
print(df_lab[df_lab["label"].isna()][["fname"]].head(30))



Distribución labels (incluye None):
label
ddos_ack_fragmentation     13
ddos_tcp_flood             11
dos_tcp_flood              11
ddos_icmp_flood            10
dos_syn_flood               8
benign                      4
dos_http_flood              2
ddos_http_flood             1
dns_spoofing                1
sql_injection_uploading     1
Name: count, dtype: int64

Ejemplos con label None (si hay):
Empty DataFrame
Columns: [fname]
Index: []


In [26]:
df_lab = df_lab.dropna(subset=["label"]).copy()

# opcional: eliminar clases con muy pocos archivos (recomendado)
min_files_per_class = 1
vc = df_lab["label"].value_counts()
keep = vc[vc >= min_files_per_class].index.tolist()
df_lab = df_lab[df_lab["label"].isin(keep)].copy()

print("Clases finales:")
print(df_lab["label"].value_counts())


Clases finales:
label
ddos_ack_fragmentation     13
ddos_tcp_flood             11
dos_tcp_flood              11
ddos_icmp_flood            10
dos_syn_flood               8
benign                      4
dos_http_flood              2
ddos_http_flood             1
dns_spoofing                1
sql_injection_uploading     1
Name: count, dtype: int64


In [27]:
import numpy as np
import pandas as pd

def reservoir_sample_csv(path, n_rows=30_000, chunksize=200_000, seed=42):
    parts = []
    for chunk in pd.read_csv(path, chunksize=chunksize, low_memory=False):
        chunk = chunk.apply(pd.to_numeric, errors="coerce")
        parts.append(chunk)
        if sum(len(p) for p in parts) >= n_rows:
            break
    out = pd.concat(parts, ignore_index=True)
    if len(out) > n_rows:
        out = out.sample(n=n_rows, random_state=seed).reset_index(drop=True)
    return out


def load_multiclass(df_files: pd.DataFrame, per_file=30_000):
    dfs = []
    for _, row in df_files.iterrows():
        fp = Path(row["file"])
        lab = row["label"]
        tmp = reservoir_sample_csv(fp, n_rows=per_file, seed=SEED)
        tmp["label"] = lab
        tmp["__source__"] = fp.name
        dfs.append(tmp)
    data = pd.concat(dfs, ignore_index=True)

    # Validación fuerte:
    if data["label"].isna().any():
        raise RuntimeError("Hay NaN en label. Revisa infer_label_from_filename y df_lab.")
    return data


df = load_multiclass(df_lab, per_file=20_000)
print("Dataset shape:", df.shape)
print(df["label"].value_counts().head(20))


Dataset shape: (1214727, 41)
label
ddos_ack_fragmentation     249482
ddos_tcp_flood             220000
dos_tcp_flood              220000
ddos_icmp_flood            200000
dos_syn_flood              160000
benign                      80000
dos_http_flood              40000
ddos_http_flood             20000
dns_spoofing                20000
sql_injection_uploading      5245
Name: count, dtype: int64


In [18]:
EPS = 1e-6

def add_features(df):
    df = df.copy()
    df = df.apply(pd.to_numeric, errors="coerce")

    # Volumen por paquete
    if "Tot sum" in df.columns and "Number" in df.columns:
        df["bytes_per_pkt"] = df["Tot sum"] / (df["Number"] + EPS)

    # Ratios TCP
    if "syn_count" in df.columns and "ack_count" in df.columns:
        df["syn_ack_ratio"] = (df["syn_count"] + 1.0) / (df["ack_count"] + 1.0)

    if "rst_count" in df.columns and "Number" in df.columns:
        df["rst_rate"] = df["rst_count"] / (df["Number"] + EPS)

    if "fin_count" in df.columns and "Number" in df.columns:
        df["fin_rate"] = df["fin_count"] / (df["Number"] + EPS)

    # “Mix” TCP/UDP (si existen)
    if "TCP" in df.columns and "UDP" in df.columns:
        df["tcp_udp_ratio"] = (df["TCP"] + EPS) / (df["UDP"] + EPS)

    # limpieza
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return df


df = add_features(df)
print("OK features. shape:", df.shape)


OK features. shape: (1729110, 46)


In [31]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold

SEED = 42
EPS = 1e-6

# 1) y y groups desde df
y_str = df["label"].astype(str).values
groups = df["__source__"].astype(str).values

# 2) X desde df (quita columnas no-feature)
drop_cols = ["label", "__source__"]
X_df = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()

# 3) todo numérico + limpieza
X_df = X_df.apply(pd.to_numeric, errors="coerce")
X_df.replace([np.inf, -np.inf], np.nan, inplace=True)
X_df = X_df.fillna(X_df.median(numeric_only=True))

# 4) clipping + log1p (si ya lo hiciste antes, puedes omitirlo, pero así queda consistente)
cont_cols = [c for c in X_df.columns if X_df[c].nunique() > 2]
p1 = X_df[cont_cols].quantile(0.01)
p99 = X_df[cont_cols].quantile(0.99)
X_df[cont_cols] = X_df[cont_cols].clip(lower=p1, upper=p99, axis=1)

skew_like = ["Rate","IAT","Tot sum","Tot size","Variance","Std","Max","AVG","Min","Header_Length","bytes_per_pkt","syn_ack_ratio","tcp_udp_ratio"]
for c in skew_like:
    if c in X_df.columns:
        X_df[c] = np.log1p(np.maximum(X_df[c], 0))

# 5) VarianceThreshold (quita columnas casi constantes)
vt = VarianceThreshold(threshold=1e-6)
X_vt = vt.fit_transform(X_df.values)
kept_cols = X_df.columns[vt.get_support()].tolist()

# 6) X final alineado
X = X_vt.astype(np.float32)

print("✅ Alineación lista")
print("len(df):", len(df), "len(X):", len(X), "len(y_str):", len(y_str), "len(groups):", len(groups))
print("features:", len(kept_cols))


✅ Alineación lista
len(df): 1214727 len(X): 1214727 len(y_str): 1214727 len(groups): 1214727
features: 31


In [32]:
from sklearn.model_selection import GroupShuffleSplit

labels = sorted(np.unique(y_str))
label_to_id = {l:i for i,l in enumerate(labels)}
id_to_label = {i:l for l,i in label_to_id.items()}
y = np.array([label_to_id[v] for v in y_str], dtype=np.int32)

print("Clases:", label_to_id)
print("N grupos:", len(np.unique(groups)))

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train = groups[train_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr2_idx, val_idx_rel = next(gss2.split(X_train, y_train, groups=groups_train))

X_tr, X_val = X_train[tr2_idx], X_train[val_idx_rel]
y_tr, y_val = y_train[tr2_idx], y_train[val_idx_rel]

print("Train:", X_tr.shape, "Val:", X_val.shape, "Test:", X_test.shape)


Clases: {'benign': 0, 'ddos_ack_fragmentation': 1, 'ddos_http_flood': 2, 'ddos_icmp_flood': 3, 'ddos_tcp_flood': 4, 'dns_spoofing': 5, 'dos_http_flood': 6, 'dos_syn_flood': 7, 'dos_tcp_flood': 8, 'sql_injection_uploading': 9}
N grupos: 62
Train: (754727, 31) Val: (200000, 31) Test: (260000, 31)


In [33]:
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

num_classes = len(labels)

# class weights (balanceo por clase)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_tr),
    y=y_tr
)
class_weight_dict = {int(c): float(w) for c, w in zip(np.unique(y_tr), class_weights)}
print("class_weight:", class_weight_dict)

# Normalización integrada (para que el modelo exportado sea autosuficiente)
norm = tf.keras.layers.Normalization(axis=-1)
norm.adapt(X_tr)

inputs = tf.keras.Input(shape=(X_tr.shape[1],), dtype=tf.float32, name="features")
x = norm(inputs)
x = tf.keras.layers.Dense(64, activation="relu")(x)
x = tf.keras.layers.Dense(32, activation="relu")(x)
x = tf.keras.layers.Dropout(0.15)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="class_probs")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc")
    ],
)

model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=6,
        restore_best_weights=True
    )
]

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=2048,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)


class_weight: {0: 1.2578783333333334, 1: 0.4453139566443634, 2: 3.773635, 3: 0.6289391666666667, 4: 0.6289391666666667, 5: 3.773635, 6: 1.8868175, 7: 0.6289391666666667, 8: 0.94340875, 9: 14.389456625357484}


2026-02-10 02:20:30.988247: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-10 02:20:33.154645: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 39105 MB memory:  -> device: 0, name: NVIDIA A40, pci bus id: 0000:17:00.0, compute capability: 8.6
2026-02-10 02:20:33.156206: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 43277 MB memory:  -> device: 1, name: NVIDIA A40, pci bus id: 0000:65:00.0, compute capability: 8.6
2026-02-10 02:20:33.157613: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 features (InputLayer)       [(None, 31)]              0         
                                                                 
 normalization (Normalizatio  (None, 31)               63        
 n)                                                              
                                                                 
 dense (Dense)               (None, 64)                2048      
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dropout (Dropout)           (None, 32)                0         
                                                                 
 class_probs (Dense)         (None, 10)                330       
                                                             

2026-02-10 02:20:53.872470: I tensorflow/stream_executor/cuda/cuda_blas.cc:1614] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


369/369 [==============================] - 3s 6ms/step - loss: 0.6828 - accuracy: 0.8241 - top3_acc: 0.9683 - val_loss: 0.7074 - val_accuracy: 0.6535 - val_top3_acc: 0.9977
Epoch 2/50
369/369 [==============================] - 1s 3ms/step - loss: 0.4208 - accuracy: 0.8823 - top3_acc: 0.9970 - val_loss: 0.7922 - val_accuracy: 0.6523 - val_top3_acc: 0.9985
Epoch 3/50
369/369 [==============================] - 2s 4ms/step - loss: 0.3878 - accuracy: 0.8911 - top3_acc: 0.9978 - val_loss: 0.8922 - val_accuracy: 0.6523 - val_top3_acc: 0.9985
Epoch 4/50
369/369 [==============================] - 1s 4ms/step - loss: 0.3727 - accuracy: 0.8943 - top3_acc: 0.9982 - val_loss: 0.9801 - val_accuracy: 0.6534 - val_top3_acc: 0.9985
Epoch 5/50
369/369 [==============================] - 2s 4ms/step - loss: 0.3648 - accuracy: 0.8962 - top3_acc: 0.9984 - val_loss: 1.0368 - val_accuracy: 0.6506 - val_top3_acc: 0.9985
Epoch 6/50
369/369 [==============================] - 2s 4ms/step - loss: 0.3583 - accuracy

In [34]:
from sklearn.metrics import classification_report, confusion_matrix

probs = model.predict(X_test, batch_size=4096)
pred = np.argmax(probs, axis=1)

print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
print("\nReport:\n", classification_report(y_test, pred, target_names=labels, digits=4))


64/64 [==============================] - 0s 1ms/step
Confusion Matrix:
 [[ 9720     0     0     0     0  3139     0     0     0  7141]
 [   23 76364    36     0     0   186  3282    11     8    90]
 [    0     0     0     0     0     0     0     0     0     0]
 [    0     5     0 19978     0     0    17     0     0     0]
 [    0    14     4     0 19046     1    37     0   894     4]
 [    0     0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0]
 [    0    13    87     0     5     9   654 39209     2    21]
 [    1   135  1061     0 66985     1   314    33 11460    10]
 [    0     0     0     0     0     0     0     0     0     0]]

Report:
                          precision    recall  f1-score   support

                 benign     0.9975    0.4860    0.6536     20000
 ddos_ack_fragmentation     0.9978    0.9546    0.9757     80000
        ddos_http_flood     0.0000    0.0000    0.0000         0
        ddos_icmp_flood   

/home/sajaimesp/miniconda3/envs/bioembed/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/sajaimesp/miniconda3/envs/bioembed/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/sajaimesp/miniconda3/envs/bioembed/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [35]:
# EXPORT_DIR = Path("export_multiclass_attack_model")
# EXPORT_DIR.mkdir(exist_ok=True)

# # orden de columnas (vital para inferencia)
# pd.Series(kept_cols).to_csv(EXPORT_DIR / "feature_order.csv", index=False, header=False)

# # mapeo clases
# with open(EXPORT_DIR / "label_map.json", "w", encoding="utf-8") as f:
#     json.dump(label_to_id, f, ensure_ascii=False, indent=2)

# model.save(EXPORT_DIR / "saved_model", include_optimizer=False)
# print("SavedModel:", (EXPORT_DIR / "saved_model").resolve())
from pathlib import Path
import pandas as pd
import json

EXPORT_DIR = Path("export_multiclass_attack_model")
EXPORT_DIR.mkdir(exist_ok=True)

# orden de columnas (vital para inferencia consistente)
pd.Series(kept_cols).to_csv(EXPORT_DIR / "feature_order.csv", index=False, header=False)

# mapeo de clases
with open(EXPORT_DIR / "label_map.json", "w", encoding="utf-8") as f:
    json.dump(label_to_id, f, ensure_ascii=False, indent=2)

# SavedModel
model.save(EXPORT_DIR / "saved_model", include_optimizer=False)
print("SavedModel:", (EXPORT_DIR / "saved_model").resolve())


INFO:tensorflow:Assets written to: export_multiclass_attack_model/saved_model/assets
SavedModel: /home/sajaimesp/Tesis_Sistemas/Data_Sets/export_multiclass_attack_model/saved_model


In [37]:
from pathlib import Path
import pandas as pd
import json

EXPORT_DIR = Path("export_multiclass_attack_model")
EXPORT_DIR.mkdir(exist_ok=True)

# 1) Guardar el orden de columnas (para inferencia consistente)
pd.Series(kept_cols).to_csv(EXPORT_DIR / "feature_order.csv", index=False, header=False)

# 2) Guardar el mapeo de clases
with open(EXPORT_DIR / "label_map.json", "w", encoding="utf-8") as f:
    json.dump(label_to_id, f, ensure_ascii=False, indent=2)

# 3) Guardar en formato Keras recomendado (TF2.13+): .keras
keras_path = EXPORT_DIR / "attack_multiclass.keras"
model.save(keras_path)
print("✅ Modelo Keras guardado en:", keras_path.resolve())

# 4) (Opcional) Guardar también como SavedModel (útil para servir / TF Serving)
savedmodel_path = EXPORT_DIR / "saved_model"
model.save(savedmodel_path, include_optimizer=False)
print("✅ SavedModel guardado en:", savedmodel_path.resolve())

# 5) (Opcional) Guardar H5 (más compatible con cosas viejas)
h5_path = EXPORT_DIR / "attack_multiclass.h5"
model.save(h5_path)
print("✅ H5 guardado en:", h5_path.resolve())


✅ Modelo Keras guardado en: /home/sajaimesp/Tesis_Sistemas/Data_Sets/export_multiclass_attack_model/attack_multiclass.keras
INFO:tensorflow:Assets written to: export_multiclass_attack_model/saved_model/assets
✅ SavedModel guardado en: /home/sajaimesp/Tesis_Sistemas/Data_Sets/export_multiclass_attack_model/saved_model
✅ H5 guardado en: /home/sajaimesp/Tesis_Sistemas/Data_Sets/export_multiclass_attack_model/attack_multiclass.h5
